# Prepare the new segmentation dataset

Run this notebook with `.venv` before `trainUnet.ipynb`. It uses `data.py` to validate pairs and save a preparation manifest consumed by `main.prepare_run`. Source images and masks are not modified. Match `DATASET` and `LIMIT` to the training notebook.


In [ ]:
import json, sys
from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from data import prepare_dataset, make_dataset
DATASET = Path('data/936')
LIMIT = 5
IMAGE_SIZE = 128
print('Python:', sys.executable)
images, masks, manifest = prepare_dataset(DATASET, LIMIT)
print('Available pairs:', len(images))
print('Selected splits:', manifest)


In [ ]:
ids = manifest['train'][:5]
fig, axes = plt.subplots(len(ids), 2, figsize=(9, 3 * len(ids)), squeeze=False)
for row, key in enumerate(ids):
    with Image.open(images[key]) as image:
        axes[row, 0].imshow(image.convert('RGB'))
    with Image.open(masks[key]) as mask:
        array = np.asarray(mask)
        print(key, 'mask IDs:', np.unique(array).tolist())
        axes[row, 1].imshow(array, cmap='viridis', vmin=0, vmax=2, interpolation='nearest')
    for col, title in enumerate((f'Image {key}', 'Ground truth')):
        axes[row, col].set_title(title)
        axes[row, col].axis('off')
plt.tight_layout()
plt.show()


In [ ]:
sample_images, sample_masks = next(iter(make_dataset(images, masks, manifest['train'],
    size=IMAGE_SIZE, batch_size=2)))
print('Image shape/dtype:', sample_images.shape, sample_images.dtype)
print('Mask shape/dtype:', sample_masks.shape, sample_masks.dtype)
print('Resized mask IDs:', np.unique(sample_masks.numpy()).tolist())
output = Path('runs')
output.mkdir(exist_ok=True)
prepared = {'dataset': str(DATASET.resolve()), 'limit': LIMIT, 'manifest': manifest}
(output / 'prepared_dataset.json').write_text(json.dumps(prepared, indent=2))
fig.savefig(output / 'prepared_dataset_preview.png', dpi=140)
print('Saved preparation manifest. Next: run trainUnet.ipynb.')
